# Cross-Trait Genetic Correlation

Goal: estimate genetic correlation between two traits from munged summary statistics and one matched LD-score reference.

This notebook uses a tiny synthetic dataset so every cell can run as a smoke test. Replace the toy tables with real `.sumstats.gz`, merged `.l2.ldscore.gz`, and `.l2.M_5_50` files for analysis.

In [ ]:
from pathlib import Path
import sys

def find_src_dir(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "ldsc").exists():
            return candidate / "src"
        nested = candidate / "ldsc_py3_restructured" / "src"
        if (nested / "ldsc").exists():
            return nested
    raise FileNotFoundError("Could not find src/ldsc from the current working directory.")

SRC_DIR = find_src_dir(Path.cwd())
REPO_DIR = SRC_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(SRC_DIR)

In [ ]:
import gzip
import os
import subprocess
import tempfile

import numpy as np
import pandas as pd

from ldsc import GlobalConfig, RegressionConfig, RegressionRunner, set_global_config
from ldsc.ldscore_calculator import LDScoreResult
from ldsc.sumstats_munger import SumstatsTable

## Python API

The public API consumes two `SumstatsTable` objects and one `LDScoreResult`. In a real run, create the sumstats with `SumstatsMunger().run(...)` or `load_sumstats(...)`, and create the LD scores with `run_ldscore(...)` or `load_ldscore_from_files(...)`.

In [ ]:
GLOBAL_CONFIG = GlobalConfig(snp_identifier="rsid", genome_build="hg38")
set_global_config(GLOBAL_CONFIG)

n_snps = 20
snps = [f"rs{i}" for i in range(1, n_snps + 1)]
index = np.arange(n_snps)

trait_1 = SumstatsTable(
    data=pd.DataFrame(
        {
            "SNP": snps,
            "Z": np.linspace(-2.0, 2.0, n_snps) + 0.1 * np.sin(index),
            "N": np.full(n_snps, 10000.0),
            "A1": ["A"] * n_snps,
            "A2": ["G"] * n_snps,
        }
    ),
    has_alleles=True,
    source_path="synthetic_trait_1",
    trait_name="trait_1",
    provenance={},
    config_snapshot=GLOBAL_CONFIG,
)

trait_2 = SumstatsTable(
    data=pd.DataFrame(
        {
            "SNP": snps,
            "Z": np.linspace(-1.5, 1.8, n_snps) + 0.2 * np.cos(index),
            "N": np.full(n_snps, 12000.0),
            "A1": ["A"] * n_snps,
            "A2": ["G"] * n_snps,
        }
    ),
    has_alleles=True,
    source_path="synthetic_trait_2",
    trait_name="trait_2",
    provenance={},
    config_snapshot=GLOBAL_CONFIG,
)

ldscore_result = LDScoreResult(
    ldscore_table=pd.DataFrame(
        {
            "CHR": ["1"] * n_snps,
            "SNP": snps,
            "BP": np.arange(100, 100 + n_snps),
            "base": np.linspace(1.0, 2.0, n_snps),
            "regr_weight": np.linspace(1.1, 1.5, n_snps),
        }
    ),
    snp_count_totals={"common_reference_snp_counts_maf_gt_0_05": np.array([100.0])},
    baseline_columns=["base"],
    query_columns=[],
    ld_reference_snps=frozenset(snps),
    ld_regression_snps=frozenset(snps),
    chromosome_results=[],
    config_snapshot=GLOBAL_CONFIG,
)

trait_1.data.head()

In [ ]:
runner = RegressionRunner(
    global_config=GLOBAL_CONFIG,
    regression_config=RegressionConfig(n_blocks=10, use_intercept=False),
)
rg = runner.estimate_rg(trait_1, trait_2, ldscore_result)

api_summary = pd.DataFrame(
    [
        {
            "trait_1": trait_1.trait_name,
            "trait_2": trait_2.trait_name,
            "rg": getattr(rg, "rg_ratio", None),
            "rg_se": getattr(rg, "rg_se", None),
            "z": getattr(rg, "z", None),
            "p": getattr(rg, "p", None),
        }
    ]
)
api_summary

## CLI

For real inputs, run `ldsc munge-sumstats` once per trait, then run `ldsc rg --sumstats-1 ... --sumstats-2 ... --ldscore ... --counts ...`. The cell below writes the toy tables to temporary LDSC artifacts and runs the same CLI path.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir_raw:
    tmpdir = Path(tmpdir_raw)
    trait_1_path = tmpdir / "trait_1.sumstats.gz"
    trait_2_path = tmpdir / "trait_2.sumstats.gz"
    ldscore_path = tmpdir / "baseline.l2.ldscore.gz"
    counts_path = tmpdir / "baseline.l2.M_5_50"
    out_prefix = tmpdir / "trait_1_trait_2"

    with gzip.open(trait_1_path, "wt", encoding="utf-8") as handle:
        trait_1.data.to_csv(handle, sep="\t", index=False)
    with gzip.open(trait_2_path, "wt", encoding="utf-8") as handle:
        trait_2.data.to_csv(handle, sep="\t", index=False)
    with gzip.open(ldscore_path, "wt", encoding="utf-8") as handle:
        ldscore_result.ldscore_table.to_csv(handle, sep="\t", index=False)
    counts_path.write_text("100\n", encoding="utf-8")

    env = os.environ.copy()
    env["PYTHONPATH"] = str(SRC_DIR)
    command = [
        sys.executable,
        "-m",
        "ldsc",
        "rg",
        "--sumstats-1",
        str(trait_1_path),
        "--sumstats-2",
        str(trait_2_path),
        "--trait-name-1",
        "trait_1",
        "--trait-name-2",
        "trait_2",
        "--ldscore",
        str(ldscore_path),
        "--counts",
        str(counts_path),
        "--count-kind",
        "m_5_50",
        "--n-blocks",
        "10",
        "--no-intercept",
        "--out",
        str(out_prefix),
    ]
    completed = subprocess.run(command, cwd=REPO_DIR, env=env, capture_output=True, text=True)
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(f"ldsc rg failed with exit code {completed.returncode}")
    cli_summary = pd.read_csv(tmpdir / "trait_1_trait_2.rg.tsv", sep="\t")

cli_summary